In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# ------------------------------------------------
# 1. READ CLUSTER CENTRES
# ------------------------------------------------

centre_file = "/Users/prernadwivedi/Downloads/21 clusters/cluster_centres_with_distance.csv"
centres = pd.read_csv(centre_file)

ra_center = dict(zip(centres["Cluster"], centres["RA_KDE_deg"]))
dec_center = dict(zip(centres["Cluster"], centres["Dec_KDE_deg"]))
distance = dict(zip(centres["Cluster"], centres["distance_pc"]))

# ------------------------------------------------
# 2. CLUSTER FILES
# ------------------------------------------------

cluster_files = {
    "NGC 2287": "/Users/prernadwivedi/Downloads/21 clusters/aptc_ngc_2287.csv",
    "NGC 2539": "/Users/prernadwivedi/Downloads/21 clusters/aptc_2539.csv",
    "NGC 7789": "/Users/prernadwivedi/Downloads/21 clusters/aptc_7789.csv"
}

# ------------------------------------------------
# 3. ANGULAR RADIUS
# ------------------------------------------------

def compute_radius(df, ra_c, dec_c):

    ra = np.radians(df["ra"])
    dec = np.radians(df["dec"])

    ra_c = np.radians(ra_c)
    dec_c = np.radians(dec_c)

    r = np.sqrt(
        ((ra-ra_c)*np.cos(dec_c))**2 +
        (dec-dec_c)**2
    )

    return np.degrees(r)


# ------------------------------------------------
# 4. MASS DENSITY PROFILE (WINDOW BINNING)
# ------------------------------------------------

def mass_density_profile(df, window=0.5, step=0.1):

    r_vals=[]
    rho_vals=[]
    err_vals=[]

    r_max=df["r_pc"].max()

    radii=np.arange(0,r_max-window,step)

    for r in radii:

        r1=r
        r2=r+window

        subset=df[(df["r_pc"]>=r1)&(df["r_pc"]<r2)]

        if len(subset)<5:
            continue

        masses=subset["mass"]

        total_mass=masses.sum()

        area=np.pi*(r2**2-r1**2)

        rho=total_mass/area

        err=np.sqrt(np.sum(masses**2))/area

        r_vals.append((r1+r2)/2)
        rho_vals.append(rho)
        err_vals.append(err)

    return np.array(r_vals),np.array(rho_vals),np.array(err_vals)


# ------------------------------------------------
# 5. PIPELINE
# ------------------------------------------------

for cluster,file in cluster_files.items():

    df=pd.read_csv(file)

    # radius
    df["r_deg"]=compute_radius(df,ra_center[cluster],dec_center[cluster])
    df["r_pc"]=np.radians(df["r_deg"])*distance[cluster]

    # remove extremely large radii (prevents vertical pile-up)
    df=df[df["r_pc"]<30]

    # absolute magnitude
    df["M_G"]=df["phot_g_mean_mag"]+5*np.log10(df["parallax"])-10

    # ------------------------------------------------
    # SPLIT INTO LOW/HIGH MASS REGIME
    # ------------------------------------------------

    break_mag=4.7

    low=df[df["M_G"]>=break_mag]
    high=df[df["M_G"]<break_mag]

    logM_low=-0.058*low["M_G"]+0.272
    logM_high=-0.108*high["M_G"]+0.501

    slope_low,intercept_low=np.polyfit(low["M_G"],logM_low,1)
    slope_high,intercept_high=np.polyfit(high["M_G"],logM_high,1)

    print(f"\n{cluster}")
    print(f"Low-mass: log10(M)= {slope_low:.3f} M_G + {intercept_low:.3f}")
    print(f"High-mass: log10(M)= {slope_high:.3f} M_G + {intercept_high:.3f}")

    # ------------------------------------------------
    # ASSIGN MASS
    # ------------------------------------------------

    logM=np.where(
        df["M_G"]>=break_mag,
        slope_low*df["M_G"]+intercept_low,
        slope_high*df["M_G"]+intercept_high
    )

    df["mass"]=10**logM

    # recreate subsets AFTER mass assignment
    low=df[df["M_G"]>=break_mag]
    high=df[df["M_G"]<break_mag]

    # ------------------------------------------------
    # MASS REGRESSION PLOT (WITH ERROR BARS)
    # ------------------------------------------------

    plt.figure(figsize=(6,4))

    y_low = slope_low*low["M_G"] + intercept_low
    y_high = slope_high*high["M_G"] + intercept_high

    sigma_low = np.std(np.log10(low["mass"]) - y_low)
    sigma_high = np.std(np.log10(high["mass"]) - y_high)

    sns.scatterplot(
        x=low["M_G"],
        y=np.log10(low["mass"]),
        s=20,
        alpha=0.35,
        color="blue"
    )

    sns.scatterplot(
        x=high["M_G"],
        y=np.log10(high["mass"]),
        s=20,
        alpha=0.35,
        color="red"
    )

    plt.errorbar(
        low["M_G"],
        np.log10(low["mass"]),
        yerr=sigma_low,
        fmt='none',
        ecolor="blue",
        alpha=0.4,
        capsize=2
    )

    plt.errorbar(
        high["M_G"],
        np.log10(high["mass"]),
        yerr=sigma_high,
        fmt='none',
        ecolor="red",
        alpha=0.4,
        capsize=2
    )

    MG_line=np.linspace(df["M_G"].min(),df["M_G"].max(),100)

    plt.plot(MG_line, slope_low*MG_line+intercept_low,
             color="blue", linewidth=2)

    plt.plot(MG_line, slope_high*MG_line+intercept_high,
             color="red", linewidth=2)

    plt.xlabel("Absolute Magnitude $M_G$")
    plt.ylabel("log10(M/M☉)")
    plt.title(cluster+" Piecewise Mass–Magnitude Relation")

    plt.tight_layout()
    plt.show()

    # ------------------------------------------------
    # MASS DENSITY PROFILES
    # ------------------------------------------------

    low_mass = df[df["mass"] < 1]
    high_mass = df[df["mass"] >= 1]

    r_low, rho_low, err_low = mass_density_profile(low_mass)
    r_high, rho_high, err_high = mass_density_profile(high_mass)

    plt.figure(figsize=(7,5))

    sns.scatterplot(x=r_low, y=rho_low, label="M < 1 M☉", s=30, color="blue")
    sns.scatterplot(x=r_high, y=rho_high, label="M ≥ 1 M☉", s=30, color="orange")

    plt.errorbar(r_low, rho_low, yerr=err_low, fmt='none', ecolor="blue", capsize=0.5)
    plt.errorbar(r_high, rho_high, yerr=err_high, fmt='none', ecolor="orange", capsize=0.5)

    plt.xscale("log")
    plt.yscale("log")

    plt.xlabel("Radius (pc)")
    plt.ylabel("Mass Density (M☉ pc$^{-2}$)")
    plt.title(cluster + " Mass Density Profile")

    plt.legend()
    plt.tight_layout()
    plt.show()

/var/folders/sp/ffpms7dn7w197xlmhjxwls700000gn/T/ipykernel_29572/187268228.py:96: DtypeWarning: Columns (182) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(file)
